In [ ]:

import cv2
import sys
import torchvision
import json
sys.path.append('..')
import torch
import lance
import os
import random
import pickle
import copy
import PIL
import time
import lance
import lance.torch.data
from typing import TypedDict
import numpy as np
from pyquaternion import Quaternion
from scipy.spatial.transform import Rotation as R
from lance.sampler import ShardedFragmentSampler

from metadrive.scenario import ScenarioDescription

from nuscenes_tools.nuscenes_utils.pipelines import Compose
import nuscenes_tools.nuscenes_utils.pipelines as pipelines
from nuscenes_tools.nuscenes_utils.box3d_instance import LiDARInstance3DBoxes

import scenarionet_tools.storage as storage

import PIL.Image as PImage
import logging

logging.getLogger("botocore").setLevel(logging.CRITICAL)
# import sys
# sys.path.append('..')
# import infinity.utils.dist as dist

creds = json.load(open('credentials.json'))['nuplan']
os.environ['AWS_ACCESS_KEY_ID'] = creds['AWS_ACCESS_KEY_ID']
os.environ['AWS_ENDPOINT_URL'] = creds['AWS_ENDPOINT_URL']
os.environ['AWS_SECRET_ACCESS_KEY'] = creds['AWS_SECRET_ACCESS_KEY']
os.environ['AWS_DEFAULT_REGION'] = creds['AWS_DEFAULT_REGION']
os.environ['PREAUTH_URL'] = creds['PREAUTH_URL']


def convert_bbox(xyz, heading, rotation, translation):
    is_torch = torch.is_tensor(xyz)
    if is_torch:
        device = xyz.device
        xyz = xyz.numpy()
        heading = heading.numpy()
        rotation = rotation.numpy()
        translation = translation.numpy()
    xyz =  xyz @ rotation.T + translation[None]
    heading = np.stack([np.zeros_like(heading), np.zeros_like(heading), heading], axis=1)
    heading = R.from_euler('xyz', heading, degrees=False).as_matrix()
    heading = rotation[None] @ heading
    heading = R.from_matrix(heading).as_euler('xyz', degrees=False)
    heading = heading[..., -1]
    
    if is_torch:
        xyz = torch.from_numpy(xyz).to(device)
        heading = torch.from_numpy(heading).to(device)
    return xyz, heading



def _fetch_next(data_iter, data_path, rank, world_size):
    if data_iter == None:
        lance_dataset = lance.torch.data.LanceDataset(
            data_path,
            columns=["scenario_id", "scenario"],
            filter="raw_sensors_valid = true",
            batch_size=1,
            # batch_readahead=8,  # Control multi-threading reads.
            to_tensor_fn=to_tensor,
            sampler = ShardedFragmentSampler(
                rank=rank,  # Rank of the current dataloader thread
                world_size=world_size,  # Total number of processes
                randomize=True,
            )
        )
        data_iter = iter(lance_dataset)
    try:
        output = next(data_iter)
    except StopIteration:
        lance_dataset = lance.torch.data.LanceDataset(
            data_path,
            columns=["scenario_id", "scenario"],
            filter="raw_sensors_valid = true",
            batch_size=1,
            # batch_readahead=8,  # Control multi-threading reads.
            to_tensor_fn=to_tensor,
            sampler = ShardedFragmentSampler(
                rank=rank,  # Rank of the current dataloader thread
                world_size=world_size,  # Total number of processes
                randomize=True,
            )
        )
        data_iter = iter(lance_dataset)
        output = next(data_iter)
    return output, data_iter



def get_nextitem(data_iter, data_path, rank, world_size):
    max_retries = 10
    for attempt in range(max_retries):
        try:
            output, data_iter = _fetch_next(data_iter, data_path, rank, world_size)
            break  # Success, exit the loop
        except (OSError, ValueError) as e:
            if "429 Too Many Requests" in str(e):
                if attempt < max_retries - 1:
                    sleep_time = (2**attempt) + random.uniform(0, 1)
                    print(
                        f"WARNING: Received 429 Too Many Requests, retrying in {sleep_time:.2f} seconds...", flush=True
                    )
                    time.sleep(sleep_time)
                else:
                    print(
                        "ERROR: Max retries reached for loading dataset from S3.", flush=True
                    )
                    raise  # Re-raise the error after max retries
            else:
                raise  # Re-raise if not a 429 error
    return output, data_iter


def to_tensor(batch, **kwargs):
    return batch


class NuPlanDataWrapper(torch.utils.data.IterableDataset):
    def __init__(self, data_config, rank=0, world_size=1, pipeline=None):
        self.data_config = data_config
        self.data_path = data_config['dataset_path']

        if pipeline is not None:
            self.pipeline = Compose(pipeline)
        else:
            self.pipeline = None
        
        self.rank = rank
        self.world_size = world_size

    def _get_scenario_info(self, scenario):
        scenario_dict = {}
        log_name = scenario['metadata']['log_name']
        date_time = log_name.split('_')[0]
        timeofday = ':'.join(date_time.split('.')[3:5])
        scenario_dict['timeofday'] = timeofday
        
        scenario_dict['location'] = scenario['metadata']['map']
        
        description = scenario['metadata']['scenario_extraction_info']['scenario_name']
        objects = scenario['metadata']['object_summary']
        sdc_id = scenario['metadata']['sdc_id']
        
        object_categories = []
        for obj_id in objects:
            if sdc_id != obj_id:
                obj_cat = objects[obj_id]['type'].lower()
                if obj_cat not in object_categories:
                    object_categories.append(obj_cat)
        object_categories = ', '.join(object_categories)
        description = description + '. ' + object_categories
        scenario_dict['description'] = description
        
        return scenario_dict
    
    def _get_object_info(self, scenario, id):
        gt_bboxes_3d = []
        gt_names_3d = []
        
        sdc_id = scenario['metadata']['sdc_id']
        tracks = scenario['tracks']
        for obj_id in tracks:
            # if obj_id == 'sdc_id':
            #     continue
            obj_info = tracks[obj_id]
            if not obj_info['state']['valid'][id]:
                continue
            position = obj_info['state']['position'][id]
            heading = obj_info['state']['heading'][id][None]
            velocity = obj_info['state']['velocity'][id]
            length = obj_info['state']['length'][id]
            width = obj_info['state']['width'][id]
            height = obj_info['state']['height'][id]
            
            obj_size = np.concatenate([length, width, height], axis=0)
            
            bbox = np.concatenate([position, obj_size, heading], axis=0)
            gt_bboxes_3d.append(bbox)

            if obj_id != 'sdc_id':
                label = obj_info['type']
            else:
                label = 'ego'
            gt_names_3d.append(label)
        
        if len(gt_bboxes_3d) > 0:
            gt_bboxes_3d = np.stack(gt_bboxes_3d, axis=0)
        else:
            gt_bboxes_3d = np.zeros((0, 7))
        gt_names_3d = np.array(gt_names_3d)

        return {'gt_bboxes_3d': gt_bboxes_3d, 'gt_names_3d': gt_names_3d}
        
    
    def _get_camera_info(self, camera_sensors):
        all_camera_info = {}
        
        for cam_name in camera_sensors:
            cam_data = camera_sensors[cam_name]
            cam_info = {}
            cam_info['data_path'] = cam_data['cam_abs_path'] # cam_abs_path in s3
            # cam_info['data_path'] = cam_info['data_path'].replace('s3://', '/media/')

            cam_info['cam_intrinsic'] = np.array(cam_data['cam_intrinsic'], dtype=np.float32)
            cam_info['sensor2lidar_translation'] = np.array(cam_data['sensor2lidar_translation'], dtype=np.float32)
            cam_info['sensor2lidar_rotation'] = np.array(cam_data['sensor2lidar_rotation'], dtype=np.float32)

            cam_info['ego2global_rotation'] = Quaternion(np.array(cam_data['ego2global_rotation'], dtype=np.float32)).rotation_matrix
            cam_info['ego2global_translation'] = np.array(cam_data['ego2global_translation'], dtype=np.float32)

            sensor2lidar = np.eye(4)
            sensor2lidar[:3, :3] = cam_info['sensor2lidar_rotation']
            sensor2lidar[:3, 3] = cam_info['sensor2lidar_translation']
            
            ego2global = np.eye(4)
            ego2global[:3, :3] = cam_info['ego2global_rotation']
            ego2global[:3, 3] = cam_info['ego2global_translation']
            
            sensor2ego = np.eye(4)
            sensor2ego[:3, :3] = Quaternion(np.array(cam_data['sensor2ego_rotation'], dtype=np.float32)).rotation_matrix
            sensor2ego[:3, 3] = np.array(cam_data['sensor2ego_translation'], dtype=np.float32)

            global2lidar = sensor2lidar @ np.linalg.inv(sensor2ego) @ np.linalg.inv(ego2global)
            cam_info['global2lidar_rotation'] = global2lidar[:3, :3]
            cam_info['global2lidar_translation'] = global2lidar[:3, 3]

            all_camera_info[cam_name] = cam_info
            

        return all_camera_info
        

    def _get_sensor_sequence_info(self, scenario):
        horizon = self.data_config['horizon']
        if horizon == 'all':
            assert not self.data_config['random_interval']
            horizon = len(scenario['raw_sensors']) // self.data_config['interval']
        intervals = []
        if self.data_config['random_interval']:
            intervals = [random.randint(0, self.data_config['interval']) for _ in range(horizon-1)]
        else:
            intervals = [self.data_config['interval']] * (horizon - 1)
        
        intervals = [0] + intervals
        
        raw_sensors = scenario['raw_sensors']
        total_len = sum(intervals)
        sids = list(range(len(raw_sensors)-total_len))
        sid = random.choice(sids)
        
        if 'first_frame' in self.data_config:
            sid = self.data_config['first_frame']
        
        timesteps = np.cumsum(intervals)
        seq_ids = timesteps + sid
        assert len(seq_ids) == horizon and seq_ids[-1] < len(raw_sensors)

        image_dict_sequence = []
        bbox_sequence = []
        for id in seq_ids:
            all_camera_info = self._get_camera_info(raw_sensors[id]['images'])
            image_dict_sequence.append(all_camera_info)
            
            if 'load_bbox' in self.data_config and self.data_config['load_bbox']:
                all_object_info = self._get_object_info(scenario, id)
                bbox_sequence.append(all_object_info)
                
        timesteps = timesteps / ((horizon - 1) * self.data_config['interval'])
        return image_dict_sequence, bbox_sequence, timesteps


    def _filter_bbox(self, xyz):
        obj_range = self.data_config['object_range']
        flag = (xyz[:, 0] > obj_range[0]) & (xyz[:, 1] > obj_range[1]) & (xyz[:, 2] > obj_range[2]) \
            & (xyz[:, 0] < obj_range[3]) & (xyz[:, 1] < obj_range[4]) & (xyz[:, 2] < obj_range[5])
        return flag

    def _convert_and_filter_bbox_coordinate(self, bbox_sequence, image_dict_sequence):
        first_frame_info = image_dict_sequence[0]
        first_frame_global2lidar_rotation = first_frame_info['CAM_F0']['global2lidar_rotation']
        first_frame_global2lidar_translation = first_frame_info['CAM_F0']['global2lidar_translation']

        for frame_box, frame_info in zip(bbox_sequence, image_dict_sequence):
            if len(frame_box['gt_bboxes_3d']) == 0:
                continue
            curr_frame_global2lidar_rotation = frame_info['CAM_F0']['global2lidar_rotation']
            curr_frame_global2lidar_translation = frame_info['CAM_F0']['global2lidar_translation']
            
            xyz = frame_box['gt_bboxes_3d'][:, :3]
            heading = frame_box['gt_bboxes_3d'][:, 6]

            curr_xyz, _ = convert_bbox(xyz, heading, curr_frame_global2lidar_rotation, curr_frame_global2lidar_translation)
            flag = self._filter_bbox(curr_xyz)
            frame_box['gt_bboxes_3d'] = frame_box['gt_bboxes_3d'][flag]
            frame_box['gt_names_3d'] = frame_box['gt_names_3d'][flag]

            if len(frame_box['gt_bboxes_3d']) == 0:
                continue
            new_xyz, new_heading = convert_bbox(frame_box['gt_bboxes_3d'][:, :3], frame_box['gt_bboxes_3d'][:, 6], first_frame_global2lidar_rotation, first_frame_global2lidar_translation)
            frame_box['gt_bboxes_3d'][:, :3] = new_xyz
            frame_box['gt_bboxes_3d'][:, 6] = new_heading
            
            # frame_box['gt_bboxes_3d'] = LiDARInstance3DBoxes(frame_box['gt_bboxes_3d'], origin=(0.5, 0.5, 0.0))

        return bbox_sequence


    def _preprocess_sample(self, item):
        scenario = pickle.loads(item['scenario'])
        scenario = ScenarioDescription(scenario)
        
        data_info = {}
        # data_info.update(scenario)
        scenario_dict = self._get_scenario_info(scenario)
        data_info.update(scenario_dict)
        data_info['sample_idx'] = item['scenario_id']
        
        
        image_dict_sequence, bbox_sequence, timesteps = self._get_sensor_sequence_info(scenario)
        for camera_infos in image_dict_sequence:
            if len(camera_infos) < self.data_config['Ncams']:
                return None
    
        if 'load_bbox' in self.data_config and self.data_config['load_bbox']:
            bbox_sequence = self._convert_and_filter_bbox_coordinate(bbox_sequence, image_dict_sequence)
    
        frame_dict_sequence = []
        for fid, camera_infos in enumerate(image_dict_sequence):
            frame_dict = {}
            frame_dict.update(data_info)
            frame_dict['img_info'] = camera_infos
            frame_dict['timestep'] = timesteps[fid]
            
            if 'load_bbox' in self.data_config and self.data_config['load_bbox']:
                frame_dict.update(bbox_sequence[fid])
            
            frame_dict_sequence.append(frame_dict)
        
        frame_dict_sequence = self._preprocess_motion(frame_dict_sequence)

        return frame_dict_sequence

    
    def _preprocess_motion(self, frame_dict_sequence):
        for id, frame_dict in enumerate(frame_dict_sequence):
            if id == 0:
                curr_to_prev_lidar_rt = np.eye(4)
            else:
                cur_img_info = frame_dict['img_info']
                cur_ego2global_rotation = cur_img_info['CAM_F0']['ego2global_rotation']
                cur_ego2global_translation = cur_img_info['CAM_F0']['ego2global_translation']
                cur_ego2global = np.eye(4)
                cur_ego2global[:3, :3] = cur_ego2global_rotation
                cur_ego2global[:3, 3] = cur_ego2global_translation

                prev_img_info = frame_dict_sequence[id-1]['img_info']
                prev_ego2global_rotation = prev_img_info['CAM_F0']['ego2global_rotation']
                prev_ego2global_translation = prev_img_info['CAM_F0']['ego2global_translation']
                prev_ego2global = np.eye(4)
                prev_ego2global[:3, :3] = prev_ego2global_rotation
                prev_ego2global[:3, 3] = prev_ego2global_translation
                
                curr_to_prev_lidar_rt = np.linalg.inv(prev_ego2global) @ cur_ego2global
            
            frame_dict['curr_to_prev_lidar_rt'] = torch.from_numpy(curr_to_prev_lidar_rt).float()
        return frame_dict_sequence
    
    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
        else:
            worker_id = 0
            num_workers = 1

        lance_iterator = None

        while True:
            sample, lance_iterator = get_nextitem(lance_iterator, self.data_config['dataset_path'], rank=self.rank*num_workers+worker_id, world_size=self.world_size*num_workers+1)


            [sample] = sample.to_pylist()
                
            sequence_data = self._preprocess_sample(sample)
            if sequence_data is None:
                # print(f"sequence_data is {sequence_data}", flush=True)
                continue
            
            try:
                if self.pipeline is not None:
                    sequence_data = [self.pipeline(item) for item in sequence_data]
            except (PIL.UnidentifiedImageError, OSError, FileNotFoundError):
                print(f"invalid data: {sample['scenario_id']}", flush=True)
                continue

            yield sequence_data
            del sequence_data

In [ ]:
import lance
import scenarionet_tools.storage as storage
from nuscenes_tools.nuscenes_utils.pipelines import Compose
import nuscenes_tools.nuscenes_utils.pipelines as pipelines
from nuscenes_tools.nuscenes_utils.box3d_instance import LiDARInstance3DBoxes

from infinity.utils.vis_utils import draw_box_on_imgs


In [ ]:

num_frames = 10
class_names = ['VEHICLE', 'CYCLIST', 'TRAFFIC_CONE', 'PEDESTRIAN', 'TRAFFIC_BARRIER', 'ego']

dataset_path = 's3://research-datasets/unified_datasets/scenarionet_lite_nuplan_full_2025_03_04/'
mean=[0.5, 0.5, 0.5]
std=[0.5, 0.5, 0.5]
img_h, img_w = 192, 336
data_config={
    'dataset_path': dataset_path,
    'horizon': num_frames,
    'interval': 5,
    'random_interval': 0,
    'first_frame': 100,
    
    'load_bbox': 1,
    'object_range': (-50, -50, -10, 50, 50, 10),

    'cams': ['CAM_F0', 'CAM_R0', 'CAM_R1', 'CAM_R2', 'CAM_B0', 'CAM_L2', 'CAM_L1', 'CAM_L0'],
    'Ncams': 8,
    #TODO: support higher resolution and bigger model
    'input_size': (img_h, img_w),
    'src_size': (1200, 2000),
    'keep_ratio': False,
    'mean': mean,
    'std': std,

    # Augmentation
    'resize': (0, 0),
    'rot': (0, 0),
    'flip': False,
    'crop_h': (0.0, 0.0),
    'resize_test':0.0,
}

lance_dataset = lance.dataset(
    dataset_path, storage_options=storage.make_s3_storage_options_from_env()
)

eval_pipeline = [
    pipelines.LoadMultiViewImageFromFiles_BEVDet(data_config, is_train=True),
    pipelines.DefaultFormatBundle3D(class_names=class_names, with_gt=True, with_label=True),
    pipelines.Collect3D(keys=['img_inputs', 'gt_bboxes_3d', 'gt_labels_3d']),
]

# build dataset
nuplan_dataset = NuPlanDataWrapper(data_config, 48, 49, eval_pipeline)

In [ ]:
OBJECT_PALETTE = [
    (255, 158, 0),
    (220, 20, 60),
    (47, 79, 79),
    (0, 0, 230),
    (112, 128, 144),
    (0, 255, 0),
]


In [ ]:
def draw_bev_bboxes(bboxes, labels, sid, tid):
    bboxes = bboxes.data
    labels = labels.data

    bboxes = bboxes.numpy()
    labels = labels.numpy()

    all_corners = []
    for bbox, label in zip(bboxes, labels):
        cx = bbox[0]
        cy = bbox[1]
        yaw = bbox[6]
        l = bbox[3]
        w = bbox[4]
        
        c1x = cx + l/2 * np.cos(yaw) - w/2 * np.sin(yaw)
        c1y = cy + l/2 * np.sin(yaw) + w/2 * np.cos(yaw)
        
        c2x = cx - l/2 * np.cos(yaw) - w/2 * np.sin(yaw)
        c2y = cy - l/2 * np.sin(yaw) + w/2 * np.cos(yaw)
        
        c3x = cx - l/2 * np.cos(yaw) + w/2 * np.sin(yaw)
        c3y = cy - l/2 * np.sin(yaw) - w/2 * np.cos(yaw)
        
        c4x = cx + l/2 * np.cos(yaw) + w/2 * np.sin(yaw)
        c4y = cy + l/2 * np.sin(yaw) - w/2 * np.cos(yaw)
        
        corners = [(c1x, c1y), (c2x, c2y), (c3x, c3y), (c4x, c4y)]
        all_corners.append(corners)
        
    all_corners = np.array(all_corners) * 5
    all_corners = all_corners + 500

    xy_min = all_corners.reshape(-1, 2).min(axis=0) - 10
    # all_corners = all_corners - xy_min[None, None]
    all_corners = all_corners.astype(np.int32)
    xy_max = all_corners.reshape(-1, 2).max(axis=0) + 10
    
    xy_max = [1000, 1000]

    x_max = int(xy_max[0])
    y_max = int(xy_max[1])

    image = np.full((y_max, x_max, 3), 255, dtype=np.uint8)
    for bid in range(len(bboxes)):
        label = int(labels[bid])
        color = OBJECT_PALETTE[label][::-1]
        
        corners = all_corners[bid]
        xs = 1000 - corners[..., 1]
        ys = 1000 - corners[..., 0]
        
        corners = np.stack([xs, ys], axis=-1)
        thick = 1
        cv2.line(image, corners[0], corners[1], color, thick)
        cv2.line(image, corners[1], corners[2], color, thick)
        cv2.line(image, corners[2], corners[3], color, thick)
        cv2.line(image, corners[0], corners[3], color, thick)

    cv2.imwrite('/home/applied/chensheng_peng/Code/cluster_training/Infinity/vis/%s_%s.png'%(sid, tid), image)

In [ ]:
import cv2
from PIL import Image

nuplan_iterator = iter(nuplan_dataset)

for i in range(0, 2):
    sample = next(nuplan_iterator)
    data = sample[0]
    img = data['img_inputs']
    # out_imgs = draw_box_on_imgs(data, img, class_names)
    # print(len(out_imgs))

In [ ]:
print(data.keys())

In [ ]:
print(data['img_metas'])

In [ ]:
def visualize_box(sample, video_imgs, num_views=8):
    all_bbox_imgs = []
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas'].data
        curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)
    
    camera_meta_data_list = []
    for frame_id in range(len(sample)):
        item = sample[frame_id]
        meta_data = item['img_metas'].data
        batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
        curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']
        rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
        # camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
        camera2lidar = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2lidar[:, :3, :3] = rot
        camera2lidar[:, :3, 3] = trans
        lidar2camera = torch.inverse(camera2lidar)
        
        camera2image = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2image[:, :3, :3] = intrins
        
        lidar2image = camera2image @ lidar2camera

        img_aug_matrix = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        img_aug_matrix[:, :3, :3] = post_rot
        img_aug_matrix[:, :3, 3] = post_trans

        camera_meta_datas = {
            'rot': rot,
            'trans': trans,
            'intrins': intrins,
            'post_rot': post_rot,
            'post_trans': post_trans,
            'seq_ids': batch_seq_ids,
            'curr_to_prev_lidar': curr_to_prev_lidar,
            'img_aug_matrix': img_aug_matrix,
            'lidar2image': lidar2image,
        }
        if num_views == 1:
            for key in camera_meta_datas:
                if key not in ['seq_ids', 'curr_to_prev_lidar']:
                    camera_meta_datas[key] = camera_meta_datas[key][:, 0:1]


        camera_meta_data_list.append(camera_meta_datas)       

    bbox_sequence = []
    for tid in range(len(sample)):
        frame_data = sample[tid]
        gt_bboxes_3d = frame_data['gt_bboxes_3d'].data
        gt_labels_3d = frame_data['gt_labels_3d'].data

        if len(gt_labels_3d) > 0:
            xyz = gt_bboxes_3d.bottom_center
            dims = gt_bboxes_3d.dims
            yaw = gt_bboxes_3d.yaw
            first_to_seq_lidar = torch.inverse(curr_to_first_lidar_list[tid])
            
            xyz, yaw = convert_bbox(xyz, yaw, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
            boxes = torch.cat([xyz, dims, yaw[:, None]], dim=-1)
            gt_bboxes_3d = LiDARInstance3DBoxes(boxes)
            bbox_num = boxes.shape[0]
        else:
            gt_bboxes_3d = LiDARInstance3DBoxes(torch.zeros((0, 7)))
            gt_labels_3d = torch.zeros(0)
            bbox_num = 0
        
        gt_labels_3d = gt_labels_3d.to(torch.int32)
        bbox_sequence.append([gt_bboxes_3d, gt_labels_3d, bbox_num])

    for tid in range(len(sample)):
        frame_meta_data = camera_meta_data_list[tid]
        frame_bbox = bbox_sequence[tid]
        
        frame_images = video_imgs[tid].permute(0, 2, 3, 1)
        frame_images = torch.unbind(frame_images, dim=0)
        frame_images = [Image.fromarray(img.cpu().numpy()) for img in frame_images]

        frame_data = {
            'gt_bboxes_3d': bbox_sequence[tid][0],
            'gt_labels_3d': bbox_sequence[tid][1],
            'lidar2image': camera_meta_data_list[tid]['lidar2image'],
            'img_aug_matrix': camera_meta_data_list[tid]['img_aug_matrix'],
        }
        
        bbox_imgs = draw_box_on_imgs(frame_data, frame_images, np.array(class_names))
        
        bbox_imgs = [torch.from_numpy(np.array(img)) for img in bbox_imgs]
        bbox_imgs = torch.stack(bbox_imgs, dim=0)
        bbox_imgs = bbox_imgs.permute(0, 3, 1, 2)
        all_bbox_imgs.append(bbox_imgs)
    
    return all_bbox_imgs

def get_gt_images(sample):
    gt_images = []
    for tid in range(len(sample)):
        frame_sample = sample[tid]
        frame_images = frame_sample['img_inputs'][0].clone()

        frame_images = (frame_images + 1) / 2
        frame_images = (frame_images*255).to(torch.uint8)
        
        gt_images.append(frame_images)
    return gt_images

In [ ]:
def save_imgs(images, suffix=''):
    TV3HW = torch.stack(images, dim=0)
    TV3HW = TV3HW.flatten(0, 1)
    vthw = torchvision.utils.make_grid(TV3HW, nrow=8, padding=0, pad_value=1.0)
    vthw = vthw.clone().permute(1, 2, 0).cpu().numpy()
    vthw = PImage.fromarray(vthw.astype(np.uint8))
    vthw.save(f"../vis/output_video_{i}{suffix}.png")
    
   

In [ ]:
import cv2
from PIL import Image

nuplan_iterator = iter(nuplan_dataset)

for i in range(0, 1):
    sample = next(nuplan_iterator)
    gt_images = get_gt_images(sample)
    
    gt_images_bbox = visualize_box(sample, gt_images)

    save_imgs(gt_images_bbox, '_gt')

AttributeError: 'Tensor' object has no attribute 'bottom_center'

In [ ]:
img = data['img_inputs']
print(len(img))

In [ ]:


import cv2
from PIL import Image

nuplan_iterator = iter(nuplan_dataset)

for i in range(0, 5):
    sample = next(nuplan_iterator)
    for tid in range(len(sample)):
        bboxes = sample[tid]['gt_bboxes_3d']
        labels = sample[tid]['gt_labels_3d']
        draw_bev_bboxes(bboxes, labels, i, tid)
        
        print(i, tid)
